In [1]:
import pandas as pd

import requests
from io import StringIO
import re
import sqlite3


### Problem-1:

You are given a SQL file link: https://drive.google.com/file/d/1WFt7B84LTHhMueoKmz8W-PRo7xXqmZf3/view?usp=share_link. Read the data by using the file and store it in a excel file. In this data, there are 3 tables named "invoices", "order_leads" and "sales_sql". So create 3 sheets to your excel file.

In [2]:
# 1. File Paths
SQL_PATH = "data/raw/supermarket.sql"
OUT_PATH = "data/processed/supermarket.xlsx"

# 2. Read raw MySQL dump file (errors='replace' prevents decoding crash)
with open(SQL_PATH, "r", encoding="utf-8", errors="replace") as f:
    sql = f.read()

# 3. Strip MySQL-specific syntax to make it SQLite compatible
sql = re.sub(r"/\*!.*?\*/;?", "", sql, flags=re.DOTALL)              # Remove conditional directives (/*! ... */)
sql = re.sub(r"^\s*(LOCK TABLES|UNLOCK TABLES).*?;", "", sql, flags=re.MULTILINE) # Remove table locking commands
sql = re.sub(r"\) ENGINE=.*?;", ");", sql)                             # Remove ENGINE & CHARSET declarations
sql = sql.replace("\\'", "''")                                         # Convert escaped quotes (\' -> '')

# 4. Load into an in-memory SQLite database
conn = sqlite3.connect(":memory:")
conn.executescript(sql)

# 5. Define explicit mapping between Database Table Names -> Excel Sheet Names
table_to_sheet = {
    "invoices": "invoices",
    "orderleads": "order_leads",
    "salesteam": "sales_sql",
}

# 6. Read tables from SQLite and export to Excel workbook
with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as writer:
    for table, sheet in table_to_sheet.items():
        df = pd.read_sql(f"SELECT * FROM `{table}`", conn)
        df.to_excel(writer, sheet_name=sheet, index=False)
        # Log row and column count for validation
        print(f"{table} -> sheet '{sheet}': {df.shape[0]} rows, {df.shape[1]} cols")

# 7. Clean up connection
conn.close()
print("Saved to", OUT_PATH)

invoices -> sheet 'invoices': 50017 rows, 8 cols
orderleads -> sheet 'order_leads': 5443 rows, 6 cols
salesteam -> sheet 'sales_sql': 4725 rows, 4 cols
Saved to data/processed/supermarket.xlsx


### Problem-2

Go to the site: https://rapidapi.com/wirefreethought/api/geodb-cities. From here, you have to grab the API and have to choose proper routes to get the cities of different countries. After getting the right API, hit that API and create a dataframe of all the cities that you can get by using the API. Then store the dataframe to a SQL. If you need to create an account or have to subscribe, then do that (it has free subscription but has some limitations. Use that free subscription and modify your accordingly to get all the data).  

In [1]:
# ==========================================================
# Problem 2 - GeoDB Cities API → SQL Database
# Using .env for API Credentials
# ==========================================================

import os
import time
import sqlite3
import requests
import pandas as pd

from dotenv import load_dotenv

# ==========================================================
# Load Environment Variables
# ==========================================================

load_dotenv()

API_KEY = os.getenv("RAPIDAPI_KEY")
API_HOST = os.getenv("RAPIDAPI_HOST")

# ==========================================================
# API Configuration
# ==========================================================

BASE_URL = "https://wft-geo-db.p.rapidapi.com/v1/geo/cities"

HEADERS = {
    "X-RapidAPI-Key": API_KEY,
    "X-RapidAPI-Host": API_HOST
}

# ==========================================================
# Countries
# Add the country codes you want to collect.
# ==========================================================

COUNTRIES = ["PK", "US", "IN", "GB", "CA", "AU", "DE", "FR", "AE", "SA"]

# ==========================================================
# API Parameters
# ==========================================================

LIMIT = 10
REQUEST_DELAY = 1.5  # 1.5 seconds wait time per request
MAX_PAGES_PER_COUNTRY = 2

# ==========================================================
# Collect Cities
# ==========================================================

all_cities = []

for country in COUNTRIES:
    print(f"Collecting cities from {country}...")
    offset = 0
    pages_fetched = 0

    while pages_fetched < MAX_PAGES_PER_COUNTRY:
        params = {
            "countryIds": country,
            "limit": LIMIT,
            "offset": offset
        }

        try:
            response = requests.get(
                BASE_URL,
                headers=HEADERS,
                params=params
            )
            response.raise_for_status()
            data = response.json()

            # "data" key holds the list of cities
            cities = data.get("data", [])

            # Agar record khatam ho jayein toh next country par move karein
            if not cities:
                break

            all_cities.extend(cities)
            
            offset += LIMIT
            pages_fetched += 1
        except requests.exceptions.RequestException as e:
            print(f"Error fetching data for {country}: {e}")
            break
        time.sleep(REQUEST_DELAY)

# ==========================================================
# Create DataFrame
# ==========================================================

df = pd.DataFrame(all_cities)

# ==========================================================
# Data Cleaning
# ==========================================================

# Add your own cleaning steps here

# Example
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

# ==========================================================
# Save to SQLite
# ==========================================================

DATABASE_NAME = "geodb_cities.db"
TABLE_NAME = "cities"

connection = sqlite3.connect(DATABASE_NAME)

df.to_sql(
    TABLE_NAME,
    connection,
    if_exists="replace",
    index=False
)

connection.close()

# ==========================================================
# Verification
# ==========================================================

print("=" * 50)
print("Collection Completed")
print("=" * 50)
print(f"Total Cities : {len(df)}")
print(f"Total Columns: {df.shape[1]}")
print(f"Database     : {DATABASE_NAME}")
print(f"Table        : {TABLE_NAME}")
print("=" * 50)

Collection Completed
Total Cities : 200
Total Columns: 13
Database     : geodb_cities.db
Table        : cities


In [2]:
df.head()

,id,wikiDataId,type,city,name,country,countryCode,region,regionCode,regionWdId,latitude,longitude,population
0,3264766,Q4663616,CITY,Abazai,Abazai,Pakistan,PK,Khyber Pakhtunkhwa,KP,Q183314,34.318611,71.593056,0
1,3282302,Q4663769,CITY,Abbas Khel Raghzai,Abbas Khel Raghzai,Pakistan,PK,Khyber Pakhtunkhwa,KP,Q183314,32.370000,69.790000,0
2,92232,Q170315,CITY,Abbottabad,Abbottabad,Pakistan,PK,Khyber Pakhtunkhwa,KP,Q183314,34.150000,73.216667,148587
3,3416977,Q306799,ADM2,Abbottabad District,Abbottabad District,Pakistan,PK,Khyber Pakhtunkhwa,KP,Q183314,34.094970,73.260430,1419072
4,3344983,Q25588910,ADM2,Abbottabad Tehsil,Abbottabad Tehsil,Pakistan,PK,Khyber Pakhtunkhwa,KP,Q183314,34.155830,73.219440,1332912
